# <b>The Skeleton Run

In [4]:
!pip install -q \
    transformers==4.45.2 \
    datasets==3.0.1 \
    peft==0.13.2 \
    trl==0.11.4 \
    accelerate==1.0.1 \
    safetensors==0.8.0 \
    tokenizers==0.20.3 \
    huggingface_hub==0.36.2 \
    numpy==1.26.4 \
    pandas==3.0.3 \
    tqdm==4.68.2

## 1. Importing Libraries

In [1]:
import os
import torch

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
)

from peft import PeftModel

## 2. Device Configuration
The system automatically uses CUDA when a GPU is available; otherwise, it falls back to CPU.
This allows the same inference pipeline to run in both GPU and CPU environments.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


## 3. Mount Google Drive
During development, trained Kaarshika models are loaded from Google Drive.
In the Flask deployment, these model paths will be replaced with the appropriate server/local model directories.

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 4. Locate the models.

In [7]:
import os
BASE_DIR = "/content/drive/MyDrive/Kaarshika/models"

### 4.1 Locate the <b>INTENT ROUTER (BERT)</b> model.

In [8]:
INTENT_MODEL_PATH = f"{BASE_DIR}/classifier_lora"

### 4.2 Locate the <b>GENERATIVE</b> model.

In [9]:
GEN_MODEL_PATH = f"{BASE_DIR}/qwen2.5-3b-lora"

### 4.3 Locate the <b>RL Agent</b>.

In [10]:
RL_AGENT_PATH = f"{BASE_DIR}/reward_model"

## 5. Load <b>INTENT ROUTER

In [14]:
INTENT_BASE_MODEL = 'distilbert-base-uncased'

intent_tokenizer = AutoTokenizer.from_pretrained(INTENT_BASE_MODEL)
intent_base_model = AutoModelForSequenceClassification.from_pretrained(INTENT_BASE_MODEL, num_labels = 2)

intent_model = PeftModel.from_pretrained(
    intent_base_model,
    INTENT_MODEL_PATH
)

intent_model.to(device)
intent_model.eval()

print("Intent Router Loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Intent Router Loaded.


In [16]:
INTENT_LABELS = {
    0: "INFORMATION",
    1: "DECISION"
}

## 6. Load Generative LLM

In [18]:
GEN_BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_BASE_MODEL)
gen_base_model = AutoModelForCausalLM.from_pretrained(
    GEN_BASE_MODEL,
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map = 'auto'
)

gen_model = PeftModel.from_pretrained(
    gen_base_model,
    GEN_MODEL_PATH
)

gen_model.eval()

print("Generative LLM Loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generative LLM Loaded.


## 7. Load <b>Reward</b> Model

In [20]:
reward_tokenizer = AutoTokenizer.from_pretrained(RL_AGENT_PATH)
reward_base_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 1)

reward_model = PeftModel.from_pretrained(
    reward_base_model,
    RL_AGENT_PATH
)

reward_model.eval()

print("Reward Model Loaded.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward Model Loaded.


---

## The HELPERS

In [17]:
def classify_intent(query):
    inputs = intent_tokenizer(query, return_tensors = 'pt', truncation = True, max_length = 512)

    with torch.no_grad():
        outputs = intent_model(**inputs)

    prediction = torch.argmax(outputs.logits, dim = -1).item()

    return INTENT_LABELS[prediction]

---